# Exercise 2: LangChain ReAct Agent as a NAT Function

In this exercise you'll learn how to:
1. Understand a pre-built LangChain ReAct agent
2. Register it as a **NeMo Agent Toolkit Function**
3. Run it through NAT's orchestration layer

## Why register agents as NAT Functions?

- **Composability** - mix agents from different frameworks in one workflow
- **Observability** - automatic tracing and profiling (Part 3)
- **Unified config** - manage everything via YAML
- **Deployment** - `nat serve` deploys the full workflow as an API

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../../.env")

## Step 1: Examine the LangChain Agent Code

Key parts of `functions.py`:

1. **Config class** - Pydantic model defining the agent's parameters
2. **`@register_function` decorator** - Tells NAT this is a composable function
3. **Builder pattern** - `_builder.get_llm()` and `_builder.get_tools()` retrieve instrumented components
4. **`FunctionInfo.from_fn()`** - Registers the callable with NAT

In [ ]:
with open("functions.py") as f:
    print(f.read())

## Step 2: Examine the YAML Config

Notice how `config.yaml` references our custom function type `langchain_research_agent`
and wires it to the Bielik LLM and tools.

In [ ]:
with open("config.yaml") as f:
    print(f.read())

## Step 3: Run the LangChain Agent via NAT

In [ ]:
!nat run --config_file config.yaml --input "Zbadaj historię sztucznej inteligencji w Polsce."

## Key Pattern: `@register_function`

```python
from nat.cli.register_workflow import register_function
from nat.builder.function_info import FunctionInfo
from nat.builder.framework_enum import LLMFrameworkEnum

@register_function(config_type=..., framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def my_agent(_config, _builder):
    llm = await _builder.get_llm(...)     # instrumented automatically
    tools = await _builder.get_tools(...)  # instrumented automatically
    
    # ... build your LangChain agent ...
    
    async def research(question: str) -> str:
        result = await agent_executor.ainvoke({"input": question})
        return result["output"]
    
    yield FunctionInfo.from_fn(research, description="...")  # Register with NAT
```

**Next:** We'll do the same with a CrewAI agent crew.